---
title: QESEM - A Qiskit Function by Qedma
description: Run quantum circuits on noisy QPUs to obtain highly accurate error-free results with highly efficient QPU-time overheads, close to fundamental bounds.
---

{/* cspell:ignore DESY Multibase, Quasicrystal, Downfolding, Aharonov */}

# QESEM: A Qiskit Function by Qedma

*See the [API reference](/docs/api/functions/qedma-qesem)*

<Admonition type="note" title="Note">
   Qiskit Functions are an experimental feature available only to IBM Quantum&reg; Premium Plan, Flex Plan, and On-Prem (via IBM Quantum Platform API) Plan users. They are in preview release status and subject to change.
</Admonition>

## Overview

While quantum processing units have vastly improved in recent years, errors due to noise and imperfections in existing hardware remain a central challenge for quantum algorithm developers. As the field approaches utility-scale quantum computations that cannot be verified classically, solutions for canceling noise with guaranteed accuracy are becoming increasingly important. To overcome this challenge, Qedma has developed Quantum Error Mitigation (QESEM), seamlessly integrated on IBM Quantum Platform as a [Qiskit Function](/docs/guides/functions).

With QESEM, users can run their quantum circuits on noisy QPUs to obtain highly accurate error-free results with highly efficient QPU-time overheads, close to fundamental bounds. To achieve this, QESEM leverages a suite of propriety methods developed by Qedma, for the characterization and reduction of errors. Error reduction techniques include gate optimization, noise-aware transpilation, error suppression (ES), and unbiased error mitigation (EM). With this combination of these characterization-based methods, users can achieve reliable, error-free results for generic large-volume quantum circuits, unlocking applications that cannot be accomplished otherwise.

For a full description of the underlying components, as well as a utility-scale demonstration, refer to the paper [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).

## Description

You can use the QESEM function by Qedma to easily estimate and execute your circuits with error suppression and mitigation, achieving larger circuit volumes and higher accuracies. To use QESEM, you provide a quantum circuit, a set of observables to measure, a target statistical accuracy for each observable, and a chosen QPU. Before you run the circuit to the target accuracy, you can estimate the required QPU time based on an analytical calculation that does not require circuit execution. Once you are satisfied with the QPU time estimation, you can execute the circuit with QESEM.

When you execute a circuit, QESEM runs a device characterization protocol tailored to your circuit, yielding a reliable noise model for the errors occurring in the circuit. Based on the characterization, QESEM first implements noise-aware transpilation to map the input circuit onto a set of physical qubits and gates, which minimizes the noise affecting the target observable. These include the natively available gates (CX/CZ on IBM&reg; devices), as well as additional gates optimized by QESEM, forming QESEM's extended gate set. QESEM then runs a set of characterization-based ES and EM circuits on the QPU and collects their measurement outcomes. These are then classically post-processed to provide an unbiased expectation value and an error bar for each observable, corresponding to the requested accuracy.

![Qedma QESEM overview](/docs/images/guides/qedma-qesem/overview.svg)

QESEM has been demonstrated to provide high-accuracy results for a variety of quantum applications and on the largest circuit volumes achievable today. QESEM offers the following user-facing features, demonstrated in the benchmarks section below:
-	**Guaranteed accuracy:** QESEM outputs unbiased estimations for expectation values of observables. Its EM method is equipped with theoretical guarantees, which - together with Qedma's cutting-edge characterization - ensure the mitigation converges to the noiseless circuit output up to the user-specified accuracy. In contrast to many heuristic EM methods that are prone to systematic errors or biases, QESEM's guaranteed accuracy is essential for ensuring reliable results in generic quantum circuits and observables.
-	**Scalability to large QPUs:** QESEM's QPU time depends on circuit volumes, but is otherwise independent of the number of qubits. Qedma has demonstrated QESEM on the largest quantum devices available today, including the IBM Quantum 127-qubit Eagle and 133-qubit Heron devices.
-	**Application-agnostic:** QESEM has been demonstrated on a variety of applications, including Hamiltonian simulation, VQE, QAOA, and amplitude estimation. Users can input any quantum circuit and observable to be measured, and obtain accurate error-free results. The only limitations are dictated by the hardware specifications and allocated QPU time, which determine the accessible circuit volumes and output accuracies. In contrast, many error reduction solutions are application-specific or involve uncontrolled heuristics, rendering them inapplicable for generic quantum circuits and applications.
-  **Extended gate set:** QESEM supports fractional-angle gates, and provides Qedma-optimized fractional-angle $Rzz(\theta)$ gates on IBM Quantum Heron and Eagle devices. This extended gate set enables more efficient compilation and unlocks circuit volumes larger by a factor of up to 2 compared to default CX/CZ compilation.
-	**Multibase observables:** QESEM supports input observables composed of many non-commuting Pauli strings, such as generic Hamiltonians. The choice of measurement bases and the optimization of QPU resource allocation (shots and circuits) is then performed automatically by QESEM to minimize the required QPU time for the requested accuracy. This optimization, which takes into account hardware fidelities and execution rates, enables you to run deeper circuits and obtain higher accuracies.

## Benchmarks

QESEM has been tested on a wide variety of use cases and applications. The following examples can assist you with assessing which types of workloads you can run with QESEM.

A key figure of merit for quantifying the hardness of both error mitigation and classical simulation for a given circuit and observable is **active volume**: the number of CNOT gates affecting the observable in the circuit. The active volume depends on the circuit depth and width, on the observable weight, and on the circuit structure, which determines the light cone of the observable. For further details, see the talk from the [2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM provides particularly large value in the high-volume regime, giving reliable results for generic circuits and observables.

![Active volume](/docs/images/guides/qedma-qesem/active_volume.svg)


| Application    | Number of qubits | Device | Circuit description | Accuracy | Total time | Runtime usage |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE circuit  | 8              | Eagle (r3) | 21 total layers, 9 measurement bases, 1D chain                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 3 steps, 2D heavy-hex topology                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 8 steps, 2D heavy-hex topology                     | 97%      | 116 min      | 23 min          |
| Trotterized Hamiltonian simulation   | 40  | Eagle (r3)            | 2 unique layers x 10 Trotter steps, 1D chain                    | 97%      | 3 hours     | 25 min         |
| Trotterized Hamiltonian simulation   | 119  | Eagle (r3)           | 3 unique layers x 9 Trotter steps, 2D heavy-hex topology                    | 95%      | 6.5 hours     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 unique layers x 15 steps, 2D heavy-hex topology                    | 99%      | 52 min             | 9 min           |

Accuracy is measured here relative to the ideal value of the observable: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, where '$\epsilon$' is the absolute precision of the mitigation (set by the user input), and $\langle O \rangle_{ideal}$ is the observable at the noiseless circuit.
'Runtime usage' measures the usage of the benchmark in batch mode (sum over usage of individual jobs), whereas 'total time' measures usage in session mode (experiment wall time), which includes additional classical and communication times. QESEM is available for execution in both modes, so that users can make the best use of their available resources.

The 28-qubit Kicked Ising circuits simulate the Discrete Time Quasicrystal studied by Shinjo et al. (see [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) and [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) on three connected loops of ibm_kawasaki. The circuit parameters taken here are $(\theta_x, \theta_z) = (0.9 \pi, 0)$, with a ferromagnetic initial state $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. The measured observable is the absolute value of the magnetization $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. The utility-scale Kicked Ising experiment was run on the 136 best qubits of ibm_fez; this particular benchmark was run at the Clifford angle $(\theta_x, \theta_z) = (\pi, 0)$, at which the active volume grows slowly with circuit depth, which - together with the high device fidelities - enables high accuracy at a short runtime.

Trotterized Hamiltonian simulation circuits are for a Transverse-Field Ising model at fractional angles: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ and $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ correspondingly (see [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). The utility-scale circuit was run on the 119 best qubits of ibm_brisbane, whereas the 40-qubit experiment was run on the best available chain. The accuracy is reported for the magnetization; high-accuracy results were obtained for higher-weight observables as well.

The VQE circuit was developed together with researchers from the Center for Quantum Technology and Applications at the Deutsches Elektronen-Synchrotron (DESY). The target observable here was a Hamiltonian consisting of a large number of non-commuting Pauli strings, emphasizing QESEM's optimized performance for multi-basis observables. Mitigation was applied to a classically-optimized ansatz; although these results are still unpublished, results of the same quality will be obtained for different circuits with similar structural properties.

## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com.

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
- Visit the [API reference](/docs/api/functions/qedma-qesem) for this Qiskit Function.
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).


</Admonition>